In [2]:
import chromadb
from langchain_groq import ChatGroq

In [3]:
with open('api_key.txt', 'r') as key:

    my_api_key = key.read().strip()

In [4]:
llm = ChatGroq(
    temperature = 0, 
    api_key = my_api_key,
    model_name = "llama-3.3-70b-versatile"
)

In [7]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://jobs.boehringer-ingelheim.com/job/Ridgefield%2C-CT-CMC-Regulatory-Data-Science-Co-Op-%28Hybrid%29-Unit/1368819133/")


page_data = loader.load().pop().page_content

print(page_data)














CMC Regulatory Data Science Co-Op (Hybrid) Job Details | BoehringerPRD
































Skip to main content
















Language 


Deutsch (Deutschland)


English (United States)


Español (España)


Français (France)


Italiano (Italia)


日本語 (日本)


Português (Brasil)


Русский язык (Россия)


简体中文 (中国大陆)





View Profile




Employee Login




























Language 


Deutsch (Deutschland)


English (United States)


Español (España)


Français (France)


Italiano (Italia)


日本語 (日本)


Português (Brasil)


Русский язык (Россия)


简体中文 (中国大陆)





View Profile




Employee Login











































                Explore our company 


Austria
Brazil
Canada (en)
Canada (fr)
China
France
Germany
Global
Japan
Mexico
Russia
South America
Spain
Taiwan
Turkey
United States of America




                Discover our careers 


Austria
Brazil
Canada (en)
Canada (fr)
China
France
Germany
Global
Japan
Mexico
Russia
South Amer

In [10]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
    """
    ### SCRAPED TEXT FROM WEBSITE: 
    {page_data}
    ### INSTRUCTION : 
    The scraped text is from the careers page of a website.
    Your job is to extract the job postings and return them in JSON format containing the following 
    keys : `role`, `experience` , `skills` and `description`.
    only return the valid JSON. 
    ### VALID JSON (NO PREAMBLE) : 

"""
)



chain_extract = prompt_extract | llm 


res = chain_extract.invoke(input = {'page_data' : page_data })

print(res.content)

```json
{
  "role": "CMC Regulatory Data Science Co-Op (Hybrid)",
  "experience": "Current undergraduate, graduate or advanced degree student in good academic standing",
  "skills": [
    "Computer Science",
    "Data Management",
    "Excellent computer and information technology literacy",
    "Excellent skills in planning, organizing, and problem-solving",
    "Strong communication (verbal and written) skills",
    "Ability to work well under pressure, work in a team environment, flexibility to adapt in a changing environment",
    "Detail oriented, well organized"
  ],
  "description": "Boehringer Ingelheim is currently seeking a talented and innovative Data Science co-op to join our CMC Regulatory Affairs department located at our Ridgefield facility. As an co-op, you will utilize your data science background to optimize data platforms and develop reports for marketed products."
}
```


In [11]:
from langchain_core.output_parsers import JsonOutputParser


json_parser = JsonOutputParser()

json_res = json_parser.parse(res.content)

json_res

{'role': 'CMC Regulatory Data Science Co-Op (Hybrid)',
 'experience': 'Current undergraduate, graduate or advanced degree student in good academic standing',
 'skills': ['Computer Science',
  'Data Management',
  'Excellent computer and information technology literacy',
  'Excellent skills in planning, organizing, and problem-solving',
  'Strong communication (verbal and written) skills',
  'Ability to work well under pressure, work in a team environment, flexibility to adapt in a changing environment',
  'Detail oriented, well organized'],
 'description': 'Boehringer Ingelheim is currently seeking a talented and innovative Data Science co-op to join our CMC Regulatory Affairs department located at our Ridgefield facility. As an co-op, you will utilize your data science background to optimize data platforms and develop reports for marketed products.'}

In [14]:
import pandas as pd


df = pd.read_csv('projects.csv')
df.head()

,TechStack,Links
0,"React, Node.js",MongoDB\thttps://example.com/react-portfolio
1,"Python, Django",PostgreSQL\thttps://example.com/ecommerce-django
2,"Vue.js, Express",MySQL\thttps://example.com/vue-dashboard
3,"Angular, Firebase",TypeScript\thttps://example.com/angular-chat-app
4,"Ruby on Rails, PostgreSQL",Redis\thttps://example.com/rails-social-network


In [16]:
import uuid

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name = 'portfolio')

if not collection.count():
    for _,row in df.iterrows():
        collection.add(documents = row['TechStack'], 
                       metadatas = {
                           'Links' : row['Links']
                       },
                       ids = [str(uuid.uuid4())])